# NLP Assignment 4 – BERT Fine Tuning on IMDB Movie Reviews

**Objective:** Fine-tune a pre-trained BERT model for binary sentiment classification using the IMDB Movie Reviews dataset.

**Dataset:** IMDB Movie Reviews (Positive / Negative Sentiment)

**Model:** `bert-base-uncased` from Hugging Face Transformers

---

### Pipeline Overview
```
Raw Data → Preprocessing → Tokenization → Model Training → Evaluation → Experiment Comparison
```

## Step 0: Install Required Libraries

In [ ]:
# Run this cell once to install dependencies
!pip install transformers datasets torch scikit-learn matplotlib seaborn -q

## Step 1: Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re
import warnings
warnings.filterwarnings('ignore')

import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from transformers import (
    BertTokenizer,
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup
)

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

from datasets import load_dataset

# Check device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## Step 2: Load Dataset

We use the **IMDB Movie Reviews** dataset loaded via Hugging Face `datasets` library. This avoids manual Kaggle download and works seamlessly in Colab/Jupyter.

- **Labels:** 0 = Negative, 1 = Positive
- **Total samples:** 50,000 (25k train + 25k test)

In [ ]:
# Load IMDB dataset from Hugging Face (equivalent to Kaggle IMDB dataset)
dataset = load_dataset('imdb')

print("Dataset structure:")
print(dataset)

# Convert to pandas DataFrames
train_df = pd.DataFrame(dataset['train'])
test_df  = pd.DataFrame(dataset['test'])

print(f"\nTraining samples : {len(train_df)}")
print(f"Test samples     : {len(test_df)}")
print(f"\nSample data:")
train_df.head(3)

## Step 3: Data Preprocessing

In [ ]:
def clean_text(text):
    """
    Clean raw review text:
    - Remove HTML tags (e.g. <br />)
    - Remove special characters and digits
    - Convert to lowercase
    - Strip extra whitespace
    """
    text = re.sub(r'<.*?>', ' ', text)          # Remove HTML tags
    text = re.sub(r'[^a-zA-Z\s]', '', text)     # Keep only letters
    text = text.lower().strip()                  # Lowercase
    text = re.sub(r'\s+', ' ', text)             # Remove extra spaces
    return text

# Apply cleaning
train_df['clean_text'] = train_df['text'].apply(clean_text)
test_df['clean_text']  = test_df['text'].apply(clean_text)

# Check for missing values
print("Missing values in train:", train_df['clean_text'].isnull().sum())
print("Missing values in test :", test_df['clean_text'].isnull().sum())

# Drop any rows where text is empty after cleaning
train_df = train_df[train_df['clean_text'].str.len() > 0].reset_index(drop=True)
test_df  = test_df[test_df['clean_text'].str.len() > 0].reset_index(drop=True)

print(f"\nCleaned Training samples : {len(train_df)}")

# Show class distribution
print("\nClass Distribution (Train):")
print(train_df['label'].value_counts())

# Visualize
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
train_df['label'].value_counts().plot(kind='bar', ax=ax[0], color=['#e74c3c','#2ecc71'], edgecolor='black')
ax[0].set_title('Label Distribution (Train)')
ax[0].set_xticklabels(['Negative (0)', 'Positive (1)'], rotation=0)
ax[0].set_ylabel('Count')

train_df['clean_text'].str.split().apply(len).hist(bins=50, ax=ax[1], color='steelblue', edgecolor='black')
ax[1].set_title('Review Length Distribution (word count)')
ax[1].set_xlabel('Word Count')
ax[1].set_ylabel('Frequency')

plt.tight_layout()
plt.savefig('data_distribution.png', dpi=150)
plt.show()
print("\nSample cleaned review:")
print(train_df['clean_text'][0][:300])

## Step 4: Data Splitting

We split the training data into **Train (80%)** and **Validation (20%)** sets. The original test split is used for final evaluation.

In [ ]:
# Use a subset for faster training (optional – remove for full training)
SAMPLE_SIZE = 5000  # Use 5000 samples for demo; set to len(train_df) for full training
TEST_SAMPLE  = 1000

train_sample = train_df.sample(n=SAMPLE_SIZE, random_state=42).reset_index(drop=True)
test_sample  = test_df.sample(n=TEST_SAMPLE, random_state=42).reset_index(drop=True)

# Train / Validation split
X_train, X_val, y_train, y_val = train_test_split(
    train_sample['clean_text'].tolist(),
    train_sample['label'].tolist(),
    test_size=0.2,
    random_state=42,
    stratify=train_sample['label']
)

X_test  = test_sample['clean_text'].tolist()
y_test  = test_sample['label'].tolist()

print(f"Train size      : {len(X_train)}")
print(f"Validation size : {len(X_val)}")
print(f"Test size       : {len(X_test)}")

## Step 5: Tokenization using `bert-base-uncased`

BERT requires:
- **Input IDs** – token indices
- **Attention Mask** – 1 for real tokens, 0 for padding
- **Max length** set to 128 (good balance of speed vs. context)

In [ ]:
MODEL_NAME = 'bert-base-uncased'
MAX_LEN    = 128

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f"Tokenizer loaded: {MODEL_NAME}")

# Quick tokenization test
sample = "This movie was absolutely fantastic! I loved every bit of it."
tokens = tokenizer(sample, max_length=MAX_LEN, truncation=True, padding='max_length', return_tensors='pt')
print(f"\nSample token ids shape : {tokens['input_ids'].shape}")
print(f"Attention mask shape   : {tokens['attention_mask'].shape}")

In [ ]:
class IMDBDataset(Dataset):
    """Custom PyTorch Dataset for IMDB reviews."""

    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts     = texts
        self.labels    = labels
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids'      : encoding['input_ids'].squeeze(0),
            'attention_mask' : encoding['attention_mask'].squeeze(0),
            'label'          : torch.tensor(self.labels[idx], dtype=torch.long)
        }

# Create Dataset objects
BATCH_SIZE = 16

train_dataset = IMDBDataset(X_train, y_train, tokenizer, MAX_LEN)
val_dataset   = IMDBDataset(X_val,   y_val,   tokenizer, MAX_LEN)
test_dataset  = IMDBDataset(X_test,  y_test,  tokenizer, MAX_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False)

print(f"Train batches      : {len(train_loader)}")
print(f"Validation batches : {len(val_loader)}")
print(f"Test batches       : {len(test_loader)}")

## Step 6: Helper Functions

Reusable functions for training, evaluation, and visualization.

In [ ]:
def train_epoch(model, loader, optimizer, scheduler, device):
    """Run one training epoch."""
    model.train()
    total_loss = 0

    for batch in loader:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['label'].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss    = outputs.loss

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # Gradient clipping
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()

    return total_loss / len(loader)


def evaluate(model, loader, device):
    """Evaluate model; return predictions and true labels."""
    model.eval()
    all_preds, all_labels = [], []
    total_loss = 0

    with torch.no_grad():
        for batch in loader:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels         = batch['label'].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            preds   = torch.argmax(outputs.logits, dim=1)

            total_loss  += outputs.loss.item()
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(loader)
    return all_preds, all_labels, avg_loss


def compute_metrics(y_true, y_pred, label=''):
    """Compute and print classification metrics."""
    acc  = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average='binary')
    rec  = recall_score(y_true, y_pred, average='binary')
    f1   = f1_score(y_true, y_pred, average='binary')

    print(f"\n{'='*40}")
    print(f" {label} – Metrics")
    print(f"{'='*40}")
    print(f"  Accuracy  : {acc:.4f}")
    print(f"  Precision : {prec:.4f}")
    print(f"  Recall    : {rec:.4f}")
    print(f"  F1 Score  : {f1:.4f}")
    print(f"\nDetailed Report:")
    print(classification_report(y_true, y_pred, target_names=['Negative', 'Positive']))

    return {'accuracy': acc, 'precision': prec, 'recall': rec, 'f1': f1}


def plot_confusion_matrix(y_true, y_pred, title='Confusion Matrix'):
    """Plot confusion matrix as heatmap."""
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Negative', 'Positive'],
                yticklabels=['Negative', 'Positive'])
    plt.title(title)
    plt.ylabel('Actual')
    plt.xlabel('Predicted')
    plt.tight_layout()
    plt.savefig(f'{title.replace(" ","_")}.png', dpi=150)
    plt.show()


def plot_training_history(train_losses, val_losses, title='Training History'):
    """Plot training and validation loss curves."""
    plt.figure(figsize=(8, 4))
    plt.plot(train_losses, label='Train Loss', marker='o')
    plt.plot(val_losses,   label='Val Loss',   marker='s')
    plt.title(title)
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(f'{title.replace(" ","_")}.png', dpi=150)
    plt.show()

print("Helper functions defined successfully.")

---

## Experiment 1: Full Fine-Tuning (All BERT Layers)

**Setup:** All BERT layers are trainable. The model learns task-specific features across the full network.

- Optimizer: AdamW
- Learning Rate: 2e-5
- Epochs: 3

In [ ]:
# ─────────────────────────────────────────────
#  EXPERIMENT 1 – Full Fine-Tuning
# ─────────────────────────────────────────────

EPOCHS      = 3
LR          = 2e-5
NUM_LABELS  = 2

# Load model
model_exp1 = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS
).to(device)

# ALL layers are trainable by default in full fine-tuning
print(f"Total trainable parameters: {sum(p.numel() for p in model_exp1.parameters() if p.requires_grad):,}")

# Optimizer and Scheduler
optimizer_exp1 = AdamW(model_exp1.parameters(), lr=LR, weight_decay=0.01)
total_steps    = len(train_loader) * EPOCHS
warmup_steps   = total_steps // 10

scheduler_exp1 = get_linear_schedule_with_warmup(
    optimizer_exp1,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

# Training loop
train_losses_exp1 = []
val_losses_exp1   = []

print("\nStarting Experiment 1: Full Fine-Tuning...\n")

for epoch in range(EPOCHS):
    train_loss = train_epoch(model_exp1, train_loader, optimizer_exp1, scheduler_exp1, device)
    preds, labels, val_loss = evaluate(model_exp1, val_loader, device)

    train_losses_exp1.append(train_loss)
    val_losses_exp1.append(val_loss)

    val_acc = accuracy_score(labels, preds)
    val_f1  = f1_score(labels, preds)

    print(f"Epoch {epoch+1}/{EPOCHS} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f} | Val F1: {val_f1:.4f}")

# Plot training history
plot_training_history(train_losses_exp1, val_losses_exp1, 'Experiment 1 – Full Fine-Tuning')

In [ ]:
# Evaluate Experiment 1 on Test Set
preds_exp1, labels_exp1, _ = evaluate(model_exp1, test_loader, device)

metrics_exp1 = compute_metrics(labels_exp1, preds_exp1, label='Experiment 1 – Full Fine-Tuning (Test Set)')
plot_confusion_matrix(labels_exp1, preds_exp1, title='Experiment 1 – Confusion Matrix')

---

## Experiment 2: Freeze All BERT Layers – Train Classifier Only

**Setup:** All BERT encoder layers are frozen. Only the classification head is trained.  
This tests how well BERT's pre-trained representations work without any task-specific adaptation.

- Fewer parameters updated → faster training
- Expected: Lower accuracy than full fine-tuning

In [ ]:
# ─────────────────────────────────────────────
#  EXPERIMENT 2 – Freeze All BERT Layers
# ─────────────────────────────────────────────

model_exp2 = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS
).to(device)

# Freeze all BERT base layers
for name, param in model_exp2.named_parameters():
    if 'classifier' not in name:  # Only classifier head remains trainable
        param.requires_grad = False

trainable = sum(p.numel() for p in model_exp2.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model_exp2.parameters())
print(f"Trainable parameters: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

optimizer_exp2 = AdamW(
    filter(lambda p: p.requires_grad, model_exp2.parameters()),
    lr=LR
)
scheduler_exp2 = get_linear_schedule_with_warmup(
    optimizer_exp2,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

train_losses_exp2 = []
val_losses_exp2   = []

print("\nStarting Experiment 2: Frozen BERT + Classifier Only...\n")

for epoch in range(EPOCHS):
    train_loss = train_epoch(model_exp2, train_loader, optimizer_exp2, scheduler_exp2, device)
    preds, labels, val_loss = evaluate(model_exp2, val_loader, device)

    train_losses_exp2.append(train_loss)
    val_losses_exp2.append(val_loss)

    val_acc = accuracy_score(labels, preds)
    val_f1  = f1_score(labels, preds)

    print(f"Epoch {epoch+1}/{EPOCHS} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f} | Val F1: {val_f1:.4f}")

plot_training_history(train_losses_exp2, val_losses_exp2, 'Experiment 2 – Frozen BERT')

In [ ]:
# Evaluate Experiment 2 on Test Set
preds_exp2, labels_exp2, _ = evaluate(model_exp2, test_loader, device)

metrics_exp2 = compute_metrics(labels_exp2, preds_exp2, label='Experiment 2 – Frozen BERT (Test Set)')
plot_confusion_matrix(labels_exp2, preds_exp2, title='Experiment 2 – Confusion Matrix')

---

## Experiment 3: Fine-Tune Last 2 BERT Layers + Classifier

**Setup:** Freeze the first 10 encoder layers. Allow the last 2 BERT encoder layers and the classifier to be updated.  
This is a balanced approach – adapts upper-level contextual representations to the task while keeping base layers stable.

- Expected: Better than frozen, possibly close to full fine-tuning

In [ ]:
# ─────────────────────────────────────────────
#  EXPERIMENT 3 – Fine-Tune Last 2 Layers
# ─────────────────────────────────────────────

model_exp3 = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS
).to(device)

# Freeze everything first
for param in model_exp3.parameters():
    param.requires_grad = False

# Unfreeze last 2 encoder layers (layers 10 and 11) and the pooler + classifier
for name, param in model_exp3.named_parameters():
    if any(layer in name for layer in ['encoder.layer.10', 'encoder.layer.11', 'pooler', 'classifier']):
        param.requires_grad = True

trainable = sum(p.numel() for p in model_exp3.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model_exp3.parameters())
print(f"Trainable parameters: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

optimizer_exp3 = AdamW(
    filter(lambda p: p.requires_grad, model_exp3.parameters()),
    lr=LR
)
scheduler_exp3 = get_linear_schedule_with_warmup(
    optimizer_exp3,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

train_losses_exp3 = []
val_losses_exp3   = []

print("\nStarting Experiment 3: Last 2 Layers Fine-Tuning...\n")

for epoch in range(EPOCHS):
    train_loss = train_epoch(model_exp3, train_loader, optimizer_exp3, scheduler_exp3, device)
    preds, labels, val_loss = evaluate(model_exp3, val_loader, device)

    train_losses_exp3.append(train_loss)
    val_losses_exp3.append(val_loss)

    val_acc = accuracy_score(labels, preds)
    val_f1  = f1_score(labels, preds)

    print(f"Epoch {epoch+1}/{EPOCHS} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f} | Val F1: {val_f1:.4f}")

plot_training_history(train_losses_exp3, val_losses_exp3, 'Experiment 3 – Last 2 Layers')

In [ ]:
# Evaluate Experiment 3 on Test Set
preds_exp3, labels_exp3, _ = evaluate(model_exp3, test_loader, device)

metrics_exp3 = compute_metrics(labels_exp3, preds_exp3, label='Experiment 3 – Last 2 Layers (Test Set)')
plot_confusion_matrix(labels_exp3, preds_exp3, title='Experiment 3 – Confusion Matrix')

---

## Step 7: Experiment Comparison & Final Analysis

In [ ]:
# ─────────────────────────────────────────────
#  COMPARISON TABLE
# ─────────────────────────────────────────────

comparison_data = {
    'Experiment': [
        'Exp 1: Full Fine-Tuning',
        'Exp 2: Frozen BERT',
        'Exp 3: Last 2 Layers'
    ],
    'Trainable Params': [
        'All (~110M)',
        'Classifier Only (~1.5K)',
        'Last 2 + Classifier (~14M)'
    ],
    'Accuracy' : [metrics_exp1['accuracy'],  metrics_exp2['accuracy'],  metrics_exp3['accuracy']],
    'Precision': [metrics_exp1['precision'], metrics_exp2['precision'], metrics_exp3['precision']],
    'Recall'   : [metrics_exp1['recall'],    metrics_exp2['recall'],    metrics_exp3['recall']],
    'F1 Score' : [metrics_exp1['f1'],        metrics_exp2['f1'],        metrics_exp3['f1']]
}

comparison_df = pd.DataFrame(comparison_data)
comparison_df = comparison_df.round(4)

print("\n" + "="*70)
print(" EXPERIMENT COMPARISON – TEST SET RESULTS")
print("="*70)
print(comparison_df.to_string(index=False))
print("="*70)

In [ ]:
# ─────────────────────────────────────────────
#  Bar Chart Comparison
# ─────────────────────────────────────────────

metrics_list = ['Accuracy', 'Precision', 'Recall', 'F1 Score']
exp_labels   = ['Full Fine-Tuning', 'Frozen BERT', 'Last 2 Layers']

x  = np.arange(len(metrics_list))
w  = 0.25

fig, ax = plt.subplots(figsize=(12, 6))

vals_e1 = [metrics_exp1['accuracy'], metrics_exp1['precision'], metrics_exp1['recall'], metrics_exp1['f1']]
vals_e2 = [metrics_exp2['accuracy'], metrics_exp2['precision'], metrics_exp2['recall'], metrics_exp2['f1']]
vals_e3 = [metrics_exp3['accuracy'], metrics_exp3['precision'], metrics_exp3['recall'], metrics_exp3['f1']]

bars1 = ax.bar(x - w,   vals_e1, w, label='Exp 1: Full Fine-Tuning', color='#3498db', edgecolor='black')
bars2 = ax.bar(x,       vals_e2, w, label='Exp 2: Frozen BERT',      color='#e74c3c', edgecolor='black')
bars3 = ax.bar(x + w,   vals_e3, w, label='Exp 3: Last 2 Layers',    color='#2ecc71', edgecolor='black')

ax.set_xlabel('Metric', fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('BERT Experiment Comparison – Test Set Metrics', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(metrics_list, fontsize=11)
ax.set_ylim(0, 1.1)
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.3)

# Annotate bars
for bars in [bars1, bars2, bars3]:
    for bar in bars:
        height = bar.get_height()
        ax.annotate(f'{height:.3f}',
                    xy=(bar.get_x() + bar.get_width() / 2, height),
                    xytext=(0, 3), textcoords="offset points",
                    ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.savefig('Experiment_Comparison.png', dpi=150)
plt.show()

---

## Step 8: Analysis and Observations

### Summary

| Experiment | Setup | Expected Behavior |
|---|---|---|
| Exp 1: Full Fine-Tuning | All ~110M parameters updated | Best performance, adapts entire model |
| Exp 2: Frozen BERT | Only classifier head updated | Fastest training, uses pre-trained features as-is |
| Exp 3: Last 2 Layers | Last 2 encoder + classifier updated | Good balance of speed and accuracy |

### Key Insights

1. **Full Fine-Tuning (Exp 1)** typically achieves the highest accuracy as the entire model adapts to the task.

2. **Frozen BERT (Exp 2)** performs surprisingly well for many NLP tasks, proving that BERT's pre-trained representations are powerful even without task-specific fine-tuning.

3. **Last 2 Layers (Exp 3)** strikes a balance – upper layers capture task-relevant patterns while the base BERT features remain stable.

4. **Training Time:** Frozen BERT trains fastest; Full fine-tuning takes the most time.

5. **Overfitting:** Full fine-tuning has higher overfitting risk on small datasets; frozen BERT is more stable.

### Recommendation
For production use on sentiment classification, **full fine-tuning or last-2-layer fine-tuning** is recommended. If compute resources are limited, **Frozen BERT** is a strong baseline.

---

## BONUS: DistilBERT with Learning Rate Scheduler

DistilBERT is a lighter, faster version of BERT – 40% smaller, 60% faster, retaining 97% of BERT's accuracy.

In [ ]:
# ─────────────────────────────────────────────
#  BONUS – DistilBERT
# ─────────────────────────────────────────────

DISTIL_MODEL = 'distilbert-base-uncased'

distil_tokenizer = AutoTokenizer.from_pretrained(DISTIL_MODEL)

# Re-create datasets with DistilBERT tokenizer
train_dataset_d = IMDBDataset(X_train, y_train, distil_tokenizer, MAX_LEN)
val_dataset_d   = IMDBDataset(X_val,   y_val,   distil_tokenizer, MAX_LEN)
test_dataset_d  = IMDBDataset(X_test,  y_test,  distil_tokenizer, MAX_LEN)

train_loader_d = DataLoader(train_dataset_d, batch_size=BATCH_SIZE, shuffle=True)
val_loader_d   = DataLoader(val_dataset_d,   batch_size=BATCH_SIZE, shuffle=False)
test_loader_d  = DataLoader(test_dataset_d,  batch_size=BATCH_SIZE, shuffle=False)

model_distil = AutoModelForSequenceClassification.from_pretrained(
    DISTIL_MODEL,
    num_labels=NUM_LABELS
).to(device)

print(f"DistilBERT parameters: {sum(p.numel() for p in model_distil.parameters()):,}")

optimizer_d = AdamW(model_distil.parameters(), lr=LR, weight_decay=0.01)
scheduler_d = get_linear_schedule_with_warmup(
    optimizer_d,
    num_warmup_steps=warmup_steps,
    num_training_steps=len(train_loader_d) * EPOCHS
)

train_losses_d = []
val_losses_d   = []

print("\nStarting BONUS: DistilBERT Full Fine-Tuning...\n")

for epoch in range(EPOCHS):
    train_loss = train_epoch(model_distil, train_loader_d, optimizer_d, scheduler_d, device)
    preds, labels, val_loss = evaluate(model_distil, val_loader_d, device)

    train_losses_d.append(train_loss)
    val_losses_d.append(val_loss)

    val_acc = accuracy_score(labels, preds)
    val_f1  = f1_score(labels, preds)
    print(f"Epoch {epoch+1}/{EPOCHS} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f} | Val F1: {val_f1:.4f}")

plot_training_history(train_losses_d, val_losses_d, 'BONUS – DistilBERT')

In [ ]:
# Evaluate DistilBERT
preds_d, labels_d, _ = evaluate(model_distil, test_loader_d, device)

metrics_distil = compute_metrics(labels_d, preds_d, label='BONUS – DistilBERT (Test Set)')
plot_confusion_matrix(labels_d, preds_d, title='BONUS DistilBERT – Confusion Matrix')

# Final comparison including DistilBERT
print("\n" + "="*75)
print(" FINAL COMPARISON INCLUDING DISTILBERT")
print("="*75)

final_data = {
    'Model'      : ['BERT Full FT', 'BERT Frozen', 'BERT Last 2L', 'DistilBERT'],
    'Accuracy'   : [metrics_exp1['accuracy'],  metrics_exp2['accuracy'],  metrics_exp3['accuracy'],  metrics_distil['accuracy']],
    'Precision'  : [metrics_exp1['precision'], metrics_exp2['precision'], metrics_exp3['precision'], metrics_distil['precision']],
    'Recall'     : [metrics_exp1['recall'],    metrics_exp2['recall'],    metrics_exp3['recall'],    metrics_distil['recall']],
    'F1 Score'   : [metrics_exp1['f1'],        metrics_exp2['f1'],        metrics_exp3['f1'],        metrics_distil['f1']]
}

final_df = pd.DataFrame(final_data).round(4)
print(final_df.to_string(index=False))

## Conclusion

This assignment demonstrated three key fine-tuning strategies for BERT on binary sentiment classification:

- **Full Fine-Tuning** provides the best accuracy by adapting all model weights to the downstream task.
- **Frozen BERT** is efficient and surprisingly effective, leveraging powerful pre-trained language representations.
- **Last 2 Layers Fine-Tuning** offers a good trade-off between compute cost and performance.
- **DistilBERT** achieves competitive results with significantly fewer parameters and faster inference.

The IMDB dataset is well-suited for sentiment analysis and BERT achieves strong results even with limited samples and epochs.
